# Coffee Review: Full Scrape and Parse
Scrapping done on 3rd March 2026

This notebook:
1. Collects all review URLs from every `review-sitemap*` listed in the sitemap index.
2. Scrapes each review page into a text file.
3. Parses the saved text into a structured CSV.


In [ ]:
import os
import re
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [ ]:
SITEMAP_INDEX_URL = "https://www.coffeereview.com/sitemap_index.xml"
SAVE_DIR = Path("coffee_reviews_text")

REQUEST_DELAY_SECONDS = 1
REQUEST_TIMEOUT_SECONDS = 30
FORCE_RESCRAPE = False

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

SAVE_DIR.mkdir(exist_ok=True)

In [ ]:
def fetch_xml(url: str) -> BeautifulSoup:
    response = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()
    return BeautifulSoup(response.content, "xml")


sitemap_soup = fetch_xml(SITEMAP_INDEX_URL)
all_sitemaps = [loc.text.strip() for loc in sitemap_soup.find_all("loc")]

review_sitemaps = [
    url for url in all_sitemaps
    if "review-sitemap" in url
]

all_review_urls = []

for sitemap_url in review_sitemaps:
    sitemap = fetch_xml(sitemap_url)
    urls = [loc.text.strip() for loc in sitemap.find_all("loc")]
    all_review_urls.extend(urls)
    print(f"Loaded {len(urls)} URLs from {sitemap_url}")
    time.sleep(REQUEST_DELAY_SECONDS)

all_review_urls = [
    url for url in all_review_urls
    if url.startswith("https://www.coffeereview.com/review/")
    and url != "https://www.coffeereview.com/review/"
    and "/wp-content/" not in url
]

# Preserve sitemap order while removing duplicates.
all_review_urls = list(dict.fromkeys(all_review_urls))

print(f"Found {len(review_sitemaps)} review sitemaps")
print(f"Found {len(all_review_urls)} unique review URLs")
all_review_urls[:5]

Loaded 1001 URLs from https://www.coffeereview.com/review-sitemap.xml
Loaded 1000 URLs from https://www.coffeereview.com/review-sitemap2.xml
Loaded 1040 URLs from https://www.coffeereview.com/review-sitemap3.xml
Loaded 1057 URLs from https://www.coffeereview.com/review-sitemap4.xml
Loaded 1055 URLs from https://www.coffeereview.com/review-sitemap5.xml
Loaded 1066 URLs from https://www.coffeereview.com/review-sitemap6.xml
Loaded 1073 URLs from https://www.coffeereview.com/review-sitemap7.xml
Loaded 1039 URLs from https://www.coffeereview.com/review-sitemap8.xml
Loaded 1082 URLs from https://www.coffeereview.com/review-sitemap9.xml
Loaded 9 URLs from https://www.coffeereview.com/review-sitemap10.xml
Found 10 review sitemaps
Found 9009 unique review URLs


['https://www.coffeereview.com/review/100-colombian/',
 'https://www.coffeereview.com/review/moka-java/',
 'https://www.coffeereview.com/review/java/',
 'https://www.coffeereview.com/review/sumatra-gayo-mountain/',
 'https://www.coffeereview.com/review/folgers-french-roast/']

In [ ]:
def url_to_filename(url: str) -> str:
    return re.sub(r"[^\w\-]", "_", url) + ".txt"


def scrape_page_text(url: str) -> str | None:
    try:
        response = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SECONDS)
        if response.status_code != 200:
            print(f"Skipped {url} (status {response.status_code})")
            return None

        soup = BeautifulSoup(response.text, "html.parser")
        page_text = soup.get_text(separator="\n", strip=True)
        return f"URL: {url}\n\n{page_text}"
    except requests.RequestException as exc:
        print(f"Request failed for {url}: {exc}")
        return None


failed_urls = []

for index, url in enumerate(all_review_urls, start=1):
    file_path = SAVE_DIR / url_to_filename(url)

    if file_path.exists() and not FORCE_RESCRAPE:
        continue

    print(f"Scraping {index}/{len(all_review_urls)}: {url}")
    text_content = scrape_page_text(url)

    if text_content is None:
        failed_urls.append(url)
    else:
        file_path.write_text(text_content, encoding="utf-8")

    time.sleep(REQUEST_DELAY_SECONDS)

print(f"Saved {len(list(SAVE_DIR.glob('*.txt')))} text files in {SAVE_DIR}")
print(f"Failed URLs: {len(failed_urls)}")
failed_urls[:10]

Scraping 1/9009: https://www.coffeereview.com/review/100-colombian/
Scraping 2/9009: https://www.coffeereview.com/review/moka-java/
Scraping 3/9009: https://www.coffeereview.com/review/java/
Scraping 4/9009: https://www.coffeereview.com/review/sumatra-gayo-mountain/
Scraping 5/9009: https://www.coffeereview.com/review/folgers-french-roast/
Scraping 6/9009: https://www.coffeereview.com/review/folger-mountain-grown-coffee/
Scraping 7/9009: https://www.coffeereview.com/review/mjb-hawaiian-blend/
Scraping 8/9009: https://www.coffeereview.com/review/traditional-premium-roast/
Scraping 9/9009: https://www.coffeereview.com/review/san-juanillo-estate-french/
Scraping 10/9009: https://www.coffeereview.com/review/sumatra/
Scraping 11/9009: https://www.coffeereview.com/review/sulawesi/
Scraping 12/9009: https://www.coffeereview.com/review/gumanch/
Scraping 13/9009: https://www.coffeereview.com/review/guatemala-antigua/
Scraping 14/9009: https://www.coffeereview.com/review/aged-sumatra-mandh-pawan

[]

In [ ]:
url_pattern = re.compile(r"^URL:\s*(https?://\S+)", re.MULTILINE)
rating_pattern = re.compile(r"^(\d{2,3})\n", re.MULTILINE)
roaster_pattern = re.compile(r"^(\d{2,3})\n(.*?)\n", re.MULTILINE | re.DOTALL)
coffee_name_pattern = re.compile(r"^(\d{2,3})\n.*?\n(.*?)\n", re.MULTILINE | re.DOTALL)
location_pattern = re.compile(r"Roaster Location:\s*(.*?)\n")
origin_pattern = re.compile(r"Coffee Origin:\s*(.*?)\n")
roast_pattern = re.compile(r"Roast Level:\s*(.*?)\n")
agtron_pattern = re.compile(r"Agtron:\s*(.*?)\n")
price_pattern = re.compile(r"Est\. Price:\s*(.*?)\n")
date_pattern = re.compile(r"Review Date:\s*(.*?)\n")
score_pattern = re.compile(
    r"(Aroma|Acidity|Acidity/Structure|Body|Flavor|Aftertaste|With Milk|Flavor in milk):\s*(\d+)"
)
section_pattern = re.compile(
    r"(Blind Assessment|Notes|Who Should Drink It|Bottom Line)\n(.*?)(?=(?:Blind Assessment|Notes|Who Should Drink It|Bottom Line|$))",
    re.DOTALL,
)


def extract_review_body(raw_text: str) -> str:
    text = raw_text.replace("\r\n", "\n")
    start_match = rating_pattern.search(text)
    start_index = start_match.start() if start_match else 0

    end_markers = ["Explore Similar Coffees", "Primary Sidebar"]
    end_positions = [text.find(marker, start_index) for marker in end_markers if text.find(marker, start_index) != -1]
    end_index = min(end_positions) if end_positions else len(text)

    return text[start_index:end_index].strip()


def extract_parsed_fields(review_body: str) -> dict:
    rating = rating_pattern.search(review_body)
    roaster = roaster_pattern.search(review_body)
    coffee_name = coffee_name_pattern.search(review_body)
    location = location_pattern.search(review_body)
    origin = origin_pattern.search(review_body)
    roast = roast_pattern.search(review_body)
    agtron = agtron_pattern.search(review_body)
    price = price_pattern.search(review_body)
    review_date = date_pattern.search(review_body)
    scores = {match[0]: int(match[1]) for match in score_pattern.findall(review_body)}
    section_data = {match[0]: match[1].strip() for match in section_pattern.findall(review_body)}

    return {
        "Rating": int(rating.group(1)) if rating else None,
        "Roaster": roaster.group(2).strip() if roaster else None,
        "Coffee Name": coffee_name.group(2).strip() if coffee_name else None,
        "Roaster Location": location.group(1).strip() if location else None,
        "Coffee Origin": origin.group(1).strip() if origin else None,
        "Roast Level": roast.group(1).strip() if roast else None,
        "Agtron": agtron.group(1).strip() if agtron else None,
        "Est. Price": price.group(1).strip() if price else None,
        "Review Date": review_date.group(1).strip() if review_date else None,
        "Aroma": scores.get("Aroma"),
        "Acidity": scores.get("Acidity"),
        "Acidity/Structure": scores.get("Acidity/Structure"),
        "Body": scores.get("Body"),
        "Flavor": scores.get("Flavor"),
        "Aftertaste": scores.get("Aftertaste"),
        "With Milk": scores.get("With Milk"),
        "Flavor in milk": scores.get("Flavor in milk"),
        "Blind Assessment": section_data.get("Blind Assessment"),
        "Notes": section_data.get("Notes"),
        "Who Should Drink It": section_data.get("Who Should Drink It"),
        "Bottom Line": section_data.get("Bottom Line"),
    }

In [ ]:
records = []

for file_path in sorted(SAVE_DIR.glob("*.txt")):
    if "wp-content" in file_path.name:
        continue

    raw_text = file_path.read_text(encoding="utf-8")
    url_match = url_pattern.search(raw_text)
    url = url_match.group(1) if url_match else None
    review_body = extract_review_body(raw_text)
    parsed_fields = extract_parsed_fields(review_body)

    record = {
        "URL": url,
        "all_text": review_body,
        **parsed_fields,
    }
    records.append(record)

result_df = pd.DataFrame(records)

result_df["Combined_Acidity"] = result_df["Acidity"].fillna(result_df["Acidity/Structure"])
result_df["With Milk"] = result_df["With Milk"].fillna(result_df["Flavor in milk"])
result_df = result_df.drop(columns=["Flavor in milk"])

result_df = result_df.drop_duplicates(subset=["URL"]).reset_index(drop=True)

raw_df = result_df[["URL", "all_text"]].copy()

print(f"Parsed {len(result_df)} unique reviews")
result_df.head()

Parsed 9009 unique reviews


,URL,all_text,Rating,Roaster,Coffee Name,Roaster Location,Coffee Origin,Roast Level,Agtron,Est. Price,...,Acidity/Structure,Body,Flavor,Aftertaste,With Milk,Blind Assessment,Notes,Who Should Drink It,Bottom Line,Combined_Acidity
0,https://www.coffeereview.com/review/100-arabic...,89\nCaffe Bomrad\n100% Arabica 100% Italiano\n...,89.0,Caffe Bomrad,100% Arabica 100% Italiano,"Torino, Italy",Not disclosed.,Medium,48/65,$54.00/1 Kilogram,...,NaN,8.0,8.0,7.0,8.0,Evaluated as espresso. Smoothly round aroma: t...,Roasted in Northern Italy and distributed in N...,A strong-charactered Northern Italian styled e...,None,NaN
1,https://www.coffeereview.com/review/100-arabic...,"87\nLucaff?\n100% Arabica, Black Label (ESE po...",87.0,Lucaff?,"100% Arabica, Black Label (ESE pod)","Padenghe sul Garda, Italy",Not disclosed.,Dark,0/80,None,...,NaN,8.0,7.0,7.0,8.0,Produced from an ESE pod on a FrancisFrancis! ...,ESE (Easy Serving Espresso) pods are wafer-lik...,An attractive pod espresso for big milk drinks.,None,NaN
2,https://www.coffeereview.com/review/100-arabic...,87\nCaribeans\n100% Arabica Coffee from Puerto...,87.0,Caribeans,100% Arabica Coffee from Puerto Rico,"San Juan, Puerto Rico","Utuado, central Puerto Rico",Medium-Light,54/69,$17.00/8 ounces,...,NaN,7.0,8.0,7.0,NaN,Bittersweet but balanced; chocolaty. Dark choc...,Produced on a single farm in the central mount...,None,Satisfying chocolate and nut notes nearly carr...,7.0
3,https://www.coffeereview.com/review/100-arabic...,88\nWaka Coffee\n100% Arabica Freeze-Dried Col...,88.0,Waka Coffee,100% Arabica Freeze-Dried Colombian (Instant C...,"Los Angeles, California",Colombia,NA,0/0,$10.99/8 single-serve packets,...,7.0,8.0,8.0,8.0,NaN,Evaluated at proportions of 5 grams of instant...,The green coffee for this product was produced...,None,A appealing 100% Colombia coffee in instant fo...,7.0
4,https://www.coffeereview.com/review/100-arabic...,72\nYuban\n100% Arabica Instant Coffee\nRoaste...,72.0,Yuban,100% Arabica Instant Coffee,"Northfield, Illinois",Colombia. All coffee of the Arabica species.,NA,0/0,$8.27/8 ounces instant,...,NaN,7.0,3.0,4.0,NaN,In the aroma caramel and wet burned wood notes...,An instant coffee evaluated as mixed in propor...,"Not good, but not the worst of the instants on...",None,4.0


In [ ]:
raw_df.to_csv('Data/coffee_review_raw_texts.csv', index=False)
result_df.to_csv('Data/coffee_reviews_parsed.csv', index=False)

print(f"Saved raw review text to Data/coffee_review_raw_texts.csv")
print(f"Saved parsed review data to Data/coffee_reviews_parsed.csv")

Saved raw review text to coffee_review_raw_texts.csv
Saved parsed review data to coffee_reviews_parsed.csv
